# 02 — Model Development
**Model Risk Governance Toolkit | Lending Club Credit Risk**

This notebook covers:
1. Preprocessing pipeline (fit on train only)
2. Logistic Regression training with credit-standard evaluation metrics
3. SHAP explainability (global importance, beeswarm, waterfall)
4. Threshold governance

**SR 11-7 context:** The model development stage must document conceptual soundness —
why this algorithm was chosen, what features are used, and how performance was evaluated.

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split

from toolkit.preprocessing import fit_and_save, transform, get_feature_names
from toolkit.model import train, evaluate, plot_roc_pr, compute_shap, plot_shap_waterfall
from toolkit.threshold_governance import run_threshold_governance

RANDOM_STATE = 42
print('Libraries loaded.')

Libraries loaded.


## 1. Load Preprocessed Splits

In [2]:
train_df   = pd.read_parquet('../data/processed/train.parquet')
monitor_df = pd.read_parquet('../data/processed/monitor.parquet')

print(f'Train:   {len(train_df):,} rows')
print(f'Monitor: {len(monitor_df):,} rows')

Train:   451,059 rows
Monitor: 894,251 rows


## 2. Train / Test Split

> **Learning annotation — Why fit transformers on train only:**
> When we fit the scaler, imputer, and encoder on the training set only, we are
> replicating what happens in production. In production, the preprocessing pipeline
> was fitted months or years ago on historical data. New loan applications are
> transformed using those same historical statistics (mean, std, median, mode).
>
> Fitting on test or monitor data is called **preprocessing leakage** — it lets
> the model see summary statistics from future data during training. This inflates
> performance estimates and causes the model to underperform in production.

In [3]:
# Stratified split: 85% model-train, 15% test (held out for evaluation and calibration)
y = train_df['default'].values

train_idx, test_idx = train_test_split(
    np.arange(len(train_df)),
    test_size=0.15,
    stratify=y,
    random_state=RANDOM_STATE
)

model_train_df = train_df.iloc[train_idx].reset_index(drop=True)
test_df        = train_df.iloc[test_idx].reset_index(drop=True)

# Further split test into test (70%) and calibration holdout (30%)
# The calibration holdout is used to fit Platt scaling in notebook 03
cal_size = int(len(test_df) * 0.30)
calib_df = test_df.iloc[:cal_size].reset_index(drop=True)
eval_df  = test_df.iloc[cal_size:].reset_index(drop=True)

print(f'Model-train: {len(model_train_df):,}')
print(f'Calibration: {len(calib_df):,}')
print(f'Evaluation:  {len(eval_df):,}')
print(f'Monitor:     {len(monitor_df):,}')

Model-train: 383,400
Calibration: 20,297
Evaluation:  47,362
Monitor:     894,251


## 3. Fit Preprocessing Pipeline

In [4]:
preprocessor, X_train = fit_and_save(
    model_train_df,
    save_path='../data/processed/preprocessor.pkl'
)
y_train = model_train_df['default'].values

X_calib  = transform(preprocessor, calib_df)
y_calib  = calib_df['default'].values

X_eval   = transform(preprocessor, eval_df)
y_eval   = eval_df['default'].values

X_monitor = transform(preprocessor, monitor_df)
y_monitor = monitor_df['default'].values

feature_names = get_feature_names(preprocessor)
print(f'Feature count: {len(feature_names)}')

[preprocessing] Fitted preprocessor on 383,400 rows, 61 features.
[preprocessing] Saved preprocessor to ../data/processed/preprocessor.pkl
Feature count: 61


## 4. Train Logistic Regression

In [5]:
model = train(X_train, y_train, save_path='../data/processed/logistic_model.pkl')

[model] Trained on 383,400 samples, 61 features.
[model] Saved model to ../data/processed/logistic_model.pkl


## 5. Performance Evaluation

> **Learning annotation — Gini and KS:**
> - **Gini coefficient** = 2 × AUC − 1. It normalises AUC to a [0, 1] scale where
>   0 = random and 1 = perfect. Banks report Gini, not AUC, because Gini has a cleaner
>   interpretation: it measures the model's ability to rank-order borrowers by risk.
>   Industry benchmarks: Gini > 0.30 = acceptable, > 0.45 = good, > 0.60 = excellent.
>
> - **KS statistic** = maximum separation between the cumulative distribution of scores
>   for defaulters vs non-defaulters. A KS of 0.40 means the score distributions of
>   the two classes are separated by 40 percentage points at the point of maximum
>   discrimination. KS > 0.20 is typically the minimum acceptable threshold for
>   a retail credit scorecard.

In [6]:
train_scores   = model.predict_proba(X_train)[:, 1]
eval_scores    = model.predict_proba(X_eval)[:, 1]
monitor_scores = model.predict_proba(X_monitor)[:, 1]

print('=== Train Performance ===')
train_metrics = evaluate(y_train, train_scores, label='Train')

print('\n=== Evaluation Set Performance ===')
eval_metrics = evaluate(y_eval, eval_scores, label='Eval')

print('\n=== Monitor Set Performance ===')
monitor_metrics = evaluate(y_monitor, monitor_scores, label='Monitor')

=== Train Performance ===
[Train] AUC-ROC=0.7001  AUC-PR=0.3164  Gini=0.4001  KS=0.2921

=== Evaluation Set Performance ===
[Eval] AUC-ROC=0.6962  AUC-PR=0.3115  Gini=0.3925  KS=0.2844

=== Monitor Set Performance ===
[Monitor] AUC-ROC=0.7142  AUC-PR=0.3959  Gini=0.4285  KS=0.3110


In [7]:
plot_roc_pr(y_eval, eval_scores, label='Test', save_dir='../data/processed')
plt.show()

[model] Saved ROC/PR plot to ..\data\processed\roc_pr_test.png


## 6. SHAP Explainability

> **Learning annotation — SHAP:**
> SHAP (SHapley Additive exPlanations) is grounded in cooperative game theory.
> For each prediction, SHAP assigns a contribution to every feature such that
> the contributions sum to the difference between the prediction and the
> average prediction. This additive decomposition is unique and satisfies
> desirable axioms (efficiency, symmetry, dummy).
>
> For regulators and model risk officers, SHAP provides:
> 1. **Global importance**: which features drive the model overall
> 2. **Individual explanation**: why a specific loan was scored as risky
> 3. **Adverse action notices**: under ECOA, denied applicants have a right
>    to know the top reasons for denial — SHAP values provide a principled
>    basis for generating those reasons automatically.

In [8]:
# Compute SHAP on a sample of the evaluation set (faster)
np.random.seed(RANDOM_STATE)
shap_idx = np.random.choice(len(X_eval), size=min(2000, len(X_eval)), replace=False)

shap_values = compute_shap(
    model,
    X_eval[shap_idx],
    feature_names=feature_names,
    save_dir='../data/processed'
)
print('SHAP plots saved.')

[model] Saved SHAP bar and beeswarm plots to ../data/processed
SHAP plots saved.


In [9]:
# Waterfall plots for individual predictions
plot_shap_waterfall(
    shap_values,
    y_true=y_eval[shap_idx],
    y_pred_proba=eval_scores[shap_idx],
    save_dir='../data/processed'
)
print('Waterfall plots saved.')

[model] Saved waterfall (True Positive) to ..\data\processed\shap_waterfall_tp.png
[model] Saved waterfall (False Positive) to ..\data\processed\shap_waterfall_fp.png
[model] Saved waterfall (False Negative) to ..\data\processed\shap_waterfall_fn.png
Waterfall plots saved.


## 7. Threshold Governance

> **Learning annotation — Threshold as a business decision:**
> Choosing a classification threshold is not a modelling task — it is a **risk
> appetite** decision made by business and risk owners, documented and approved
> through governance. The model developer should present the trade-off surface
> (this sweep), but the final threshold must be chosen by someone accountable
> for business outcomes.
>
> SR 11-7 Section V requires that "assumptions and limitations" — including
> threshold selection — be documented in the validation report. A threshold
> that maximises F1 may be fine for a balanced credit policy, but a bank in
> a stressed capital position might mandate a conservative threshold (high F2)
> regardless of approval rate impact.

In [10]:
threshold_results = run_threshold_governance(
    y_true=y_eval,
    y_score=eval_scores,
    save_dir='../data/processed'
)

candidates = threshold_results['candidates']
print('\n=== Three Candidate Thresholds ===')
for name, info in candidates.items():
    print(f"  {name:12s}: threshold={info['threshold']:.2f} | "
          f"approval={info['approval_rate']:.1%} | "
          f"default_rate_approved={info['default_rate_approved']:.1%}")

[threshold] Candidate thresholds:
  conservative: threshold=0.37  approval=32.9%  default_rate_approved=7.0%
  balanced    : threshold=0.51  approval=63.3%  default_rate_approved=10.7%
  liberal     : threshold=0.90  approval=99.9%  default_rate_approved=16.9%
[threshold] Saved sweep plot to ..\data\processed\threshold_sweep.png
[threshold] Saved threshold sweep table to ../data/processed/threshold_sweep.csv

=== Three Candidate Thresholds ===
  conservative: threshold=0.37 | approval=32.9% | default_rate_approved=7.0%
  balanced    : threshold=0.51 | approval=63.3% | default_rate_approved=10.7%
  liberal     : threshold=0.90 | approval=99.9% | default_rate_approved=16.9%


In [11]:
# Save all intermediate outputs for downstream notebooks
import pickle

np.save('../data/processed/train_scores.npy',   train_scores)
np.save('../data/processed/eval_scores.npy',    eval_scores)
np.save('../data/processed/monitor_scores.npy', monitor_scores)
np.save('../data/processed/y_eval.npy',         y_eval)
np.save('../data/processed/y_monitor.npy',      y_monitor)
np.save('../data/processed/y_train.npy',        y_train)
np.save('../data/processed/X_eval.npy',         X_eval)
np.save('../data/processed/X_calib.npy',        X_calib)
np.save('../data/processed/y_calib.npy',        y_calib)

import json
metrics_out = {
    'train':   train_metrics,
    'test':    eval_metrics,
    'monitor': monitor_metrics,
    'threshold_candidates': candidates
}
with open('../data/processed/metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)

print('All intermediate outputs saved to data/processed/')

All intermediate outputs saved to data/processed/
